# Task 10 — LMR v2.1 Temporal/Spatial Structure and Grid Behaviour

Characterise the LMR v2.1 reconstruction before any signature design: variables, grid, ensemble structure, variance decomposition, Band C sanity check, and L8→LMR cell mapping.

In [1]:
# Cell 2 — Imports and config
import numpy as np
import pandas as pd
import netCDF4 as nc
import matplotlib.pyplot as plt
from pathlib import Path
import sys, time as _time
sys.path.insert(0, "/Users/karlg/Documents/Repos/_cedop")
from scripts.shared.db_utils import db_connect

LMR_DIR = Path("/Users/karlg/Documents/Repos/_cedop/data/lmr_v2.1")
OUT_DIR = Path("/Users/karlg/Documents/Repos/_cedop/output/edop/explore")

MEAN_FILES = {
    "pdsi":  LMR_DIR / "pdsi_MCruns_ensemble_mean_LMRv2.1.nc",
    "air":   LMR_DIR / "air_MCruns_ensemble_mean_LMRv2.1.nc",
    "prate": LMR_DIR / "prate_MCruns_ensemble_mean_LMRv2.1.nc",
}
SPREAD_FILES = {
    "pdsi":  LMR_DIR / "pdsi_MCruns_ensemble_spread_LMRv2.1.nc",
    "air":   LMR_DIR / "air_MCruns_ensemble_spread_LMRv2.1.nc",
    "prate": LMR_DIR / "prate_MCruns_ensemble_spread_LMRv2.1.nc",
}
NHMT_FILE = LMR_DIR / "nhmt_MCruns_ensemble_full_LMRv2.1.nc"
GMT_FILE  = LMR_DIR / "gmt_MCruns_ensemble_full_LMRv2.1.nc"

print("Files present:")
for f in sorted(LMR_DIR.glob("*.nc")):
    print(f"  {f.name}  ({f.stat().st_size/1e6:.0f} MB)")

Files present:
  air_MCruns_ensemble_mean_LMRv2.1.nc  (2058 MB)
  air_MCruns_ensemble_spread_LMRv2.1.nc  (363 MB)
  gmt_MCruns_ensemble_full_LMRv2.1.nc  (13 MB)
  nhmt_MCruns_ensemble_full_LMRv2.1.nc  (13 MB)
  pdsi_MCruns_ensemble_mean_LMRv2.1.nc  (1034 MB)
  pdsi_MCruns_ensemble_spread_LMRv2.1.nc  (186 MB)
  prate_MCruns_ensemble_mean_LMRv2.1.nc  (2138 MB)
  prate_MCruns_ensemble_spread_LMRv2.1.nc  (372 MB)


In [2]:
# Cell 3 — File structure inspection
for var, path in MEAN_FILES.items():
    ds = nc.Dataset(path)
    print(f"=== {var} (mean) ===")
    for dim in ds.dimensions.values():
        print(f"  dim {dim.name}: {dim.size}")
    for v in ds.variables.values():
        units = getattr(v, "units", "")
        lname = getattr(v, "long_name", "")
        fill  = getattr(v, "_FillValue", None)
        print(f"  var {v.name}: {v.shape}  units={units}  fill={fill}")
    ds.close()

ds = nc.Dataset(SPREAD_FILES["pdsi"])
print("=== pdsi (spread) — structure check ===")
print("  dims:", {d.name: d.size for d in ds.dimensions.values()})
for v in ds.variables.values():
    print(f"  var {v.name}: {v.shape}")
ds.close()

ds = nc.Dataset(NHMT_FILE)
print("=== nhmt (full) ===")
for v in ds.variables.values():
    print(f"  var {v.name}: {v.shape}  {getattr(v,'units','')}")
ds.close()

=== pdsi (mean) ===
  dim time: 2001
  dim MCrun: 20
  dim lat: 91
  dim lon: 180
  var time: (2001,)  units=days since 0000-01-01 00:00:00  fill=None
  var lat: (91,)  units=degrees_north  fill=None
  var lon: (180,)  units=degrees_east  fill=None
  var pdsi: (2001, 20, 91, 180)  units=  fill=-9.969209968386869e+36
=== air (mean) ===
  dim time: 2001
  dim MCrun: 20
  dim lat: 91
  dim lon: 180
  var time: (2001,)  units=days since 0000-01-01 00:00:00  fill=None
  var lat: (91,)  units=degrees_north  fill=None
  var lon: (180,)  units=degrees_east  fill=None
  var air: (2001, 20, 91, 180)  units=degK  fill=-9.969209968386869e+36
=== prate (mean) ===
  dim time: 2001
  dim MCrun: 20
  dim lat: 91
  dim lon: 180
  var time: (2001,)  units=days since 0000-01-01 00:00:00  fill=None
  var lat: (91,)  units=degrees_north  fill=None
  var lon: (180,)  units=degrees_east  fill=None
  var prate: (2001, 20, 91, 180)  units=kg/m^2/s  fill=-9.969209968386869e+36
=== pdsi (spread) — structure chec

In [4]:
# Cell 4 — Grid and time axis
ds = nc.Dataset(MEAN_FILES["pdsi"])
lats = ds.variables["lat"][:]
lons = ds.variables["lon"][:]
time_var = ds.variables["time"]
times = nc.num2date(time_var[:], time_var.units, has_year_zero=True)
years = np.array([t.year for t in times])

# Land mask from first time step, first MCrun
pdsi_t0 = np.ma.filled(ds.variables["pdsi"][0, 0, :, :], np.nan)
ds.close()

FILL_THRESH = 1e10
land_mask = np.isfinite(pdsi_t0) & (np.abs(pdsi_t0) < FILL_THRESH)
n_total = pdsi_t0.size
n_valid = int(land_mask.sum())

print(f"Grid: {len(lats)} lat x {len(lons)} lon = {n_total:,} total cells")
print(f"  Lat: {lats.min():.1f} to {lats.max():.1f} deg  step={np.diff(lats).mean():.2f} deg")
print(f"  Lon: {lons.min():.1f} to {lons.max():.1f} deg  step={np.diff(lons).mean():.2f} deg")
print(f"Valid land cells: {n_valid:,}  ({100*n_valid/n_total:.1f}%)")
print(f"\nTime axis: {len(years)} steps, {years[0]}-{years[-1]} CE")
print(f"MCrun dimension: 20 members")
print(f"Full array size uncompressed: {2001*20*91*180*4/1e9:.2f} GB — never load whole array")

Grid: 91 lat x 180 lon = 16,380 total cells
  Lat: -90.0 to 90.0 deg  step=2.00 deg
  Lon: 0.0 to 358.0 deg  step=2.00 deg
Valid land cells: 16,380  (100.0%)

Time axis: 2001 steps, 0-1998 CE
MCrun dimension: 20 members
Full array size uncompressed: 2.62 GB — never load whole array


In [5]:
# Cell 5 — Grid map (LMR covers all cells; no land/ocean masking in the data)
lon_grid, lat_grid = np.meshgrid(lons, lats)
fig, ax = plt.subplots(figsize=(14, 5))
ax.scatter(lon_grid.ravel(), lat_grid.ravel(), s=2, c="steelblue", alpha=0.5)
ax.set_title(f"LMR v2.1 grid — {n_valid:,} cells at 2x2 deg (values provided at all cells incl. ocean)")
ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
ax.set_xlim(0, 360); ax.set_ylim(-90, 90)
plt.tight_layout()
plt.savefig(OUT_DIR / "10_lmr_land_grid.png", dpi=120)
plt.show()
print("Note: LMR lon axis runs 0-358 (not -180 to 180) — keep in mind for lookups")
print("Saved 10_lmr_land_grid.png")

Note: LMR lon axis runs 0-358 (not -180 to 180) — keep in mind for lookups
Saved 10_lmr_land_grid.png


In [6]:
# Cell 6 — Sample location selection (stratified by latitude band, anchored to L8 basin centroids)
# Use L8 centroids snapped to nearest LMR cell to guarantee land locations
BANDS = [
    ("Arctic",         60,  90,  4),
    ("Temperate NH",   30,  60, 10),
    ("Subtropical NH", 15,  30,  5),
    ("Tropical N",      0,  15,  4),
    ("Tropical S",    -15,   0,  4),
    ("Subtropical SH",-30, -15,  3),
    ("Temperate SH",  -60, -30,  4),
]
BAND_COLORS = {
    "Arctic": "#1f77b4", "Temperate NH": "#ff7f0e", "Subtropical NH": "#2ca02c",
    "Tropical N": "#d62728", "Tropical S": "#9467bd",
    "Subtropical SH": "#8c564b", "Temperate SH": "#e377c2"
}

# Load L8 centroids, snap to LMR grid (lon 0-360, lat -90 to 90, step 2)
conn = db_connect()
centroids = pd.read_sql(
    "SELECT ST_X(ST_Centroid(geom)) AS lon, ST_Y(ST_Centroid(geom)) AS lat FROM public.basin08",
    conn)
conn.close()

# Snap to nearest LMR node; convert lon to 0-360
centroids["lmr_lon"] = np.round(centroids["lon"].values % 360 / 2) * 2
centroids["lmr_lat"] = np.round(centroids["lat"].values / 2) * 2
# Get unique LMR cells (deduplicate — many basins per cell)
lmr_cells = centroids[["lmr_lat","lmr_lon"]].drop_duplicates().reset_index(drop=True)

# Find lat/lon index into LMR arrays for each candidate cell
def lmr_indices(clat, clon):
    li = int(np.argmin(np.abs(lats - clat)))
    lj = int(np.argmin(np.abs(lons - clon)))
    return li, lj

np.random.seed(42)
sample_cells = []
for band_name, lat_lo, lat_hi, n in BANDS:
    cands = lmr_cells[(lmr_cells.lmr_lat >= lat_lo) & (lmr_cells.lmr_lat < lat_hi)]
    chosen = cands.sample(min(n, len(cands)), random_state=42)
    for _, row in chosen.iterrows():
        li, lj = lmr_indices(row.lmr_lat, row.lmr_lon)
        sample_cells.append({
            "band": band_name, "lat": float(row.lmr_lat), "lon": float(row.lmr_lon),
            "lat_idx": li, "lon_idx": lj,
        })

sample_df = pd.DataFrame(sample_cells).reset_index(drop=True)
print(f"Sample: {len(sample_df)} locations (L8-anchored, deduplicated LMR cells)")
print(sample_df.groupby("band").size().to_string())

Sample: 34 locations (L8-anchored, deduplicated LMR cells)
band
Arctic             4
Subtropical NH     5
Subtropical SH     3
Temperate NH      10
Temperate SH       4
Tropical N         4
Tropical S         4


In [7]:
# Cell 7 — Extract sample time series (lazy load — slice per cell, never full array)
def extract_cell_series(nc_path, var_name, li, lj):
    """Return (grand_mean_ts, std_across_ts) for one grid cell. Shape: (2001,)."""
    ds = nc.Dataset(nc_path)
    data = np.ma.filled(ds.variables[var_name][:, :, li, lj], np.nan).astype(np.float32)
    ds.close()
    return np.nanmean(data, axis=1), np.nanstd(data, axis=1)

print("Extracting sample time series (mean + spread per cell)...")
t0 = _time.time()
series = {var: [] for var in ["pdsi", "air", "prate"]}

for var in ["pdsi", "air", "prate"]:
    print(f"  {var}...", end="", flush=True)
    for _, row in sample_df.iterrows():
        li, lj = int(row.lat_idx), int(row.lon_idx)
        mean_ts, std_across = extract_cell_series(MEAN_FILES[var],   var, li, lj)
        spread_ts, _        = extract_cell_series(SPREAD_FILES[var], var, li, lj)
        series[var].append({"mean": mean_ts, "std_across": std_across, "spread": spread_ts})
    print(" done")

print(f"Done in {_time.time()-t0:.1f}s")

 done
Done in 38.2s


In [8]:
# Cell 8 — Time series gallery (ensemble grand mean, all sample locations)
UNIT_LABELS = {"pdsi": "PDSI", "air": "Temperature (K)", "prate": "Precip rate (kg/m2/s)"}

fig, axes = plt.subplots(3, 1, figsize=(16, 11), sharex=True)
for ax, var in zip(axes, ["pdsi", "air", "prate"]):
    for i, row in sample_df.iterrows():
        color = BAND_COLORS[row["band"]]
        ax.plot(years, series[var][i]["mean"], lw=0.5, alpha=0.5, color=color)
    ax.set_ylabel(UNIT_LABELS[var], fontsize=9)
    for yr in [500, 1000, 1500]:
        ax.axvline(yr, color="#cccccc", lw=0.7, ls="--")

from matplotlib.lines import Line2D
handles = [Line2D([0],[0], color=BAND_COLORS[b], lw=1.5, label=b) for b in BAND_COLORS]
axes[0].legend(handles=handles, loc="upper left", fontsize=7, ncol=2)
axes[-1].set_xlabel("Year CE")
plt.suptitle("LMR v2.1 — ensemble grand mean at sample locations (coloured by latitude band)",
             fontsize=11)
plt.tight_layout()
plt.savefig(OUT_DIR / "10_lmr_sample_timeseries.png", dpi=120)
plt.show()
print("Saved 10_lmr_sample_timeseries.png")

Saved 10_lmr_sample_timeseries.png


In [9]:
# Cell 9 — Variance decomposition: geographic vs temporal
rows = []
for var in ["pdsi", "air", "prate"]:
    arr = np.array([s["mean"] for s in series[var]])   # (n_locs, 2001)
    total_var = np.nanvar(arr)
    geo_var   = np.nanvar(np.nanmean(arr, axis=1))      # variance of location means
    time_var  = np.nanmean(np.nanvar(arr, axis=1))      # mean within-location temporal variance
    rows.append({
        "variable": var, "total_var": total_var,
        "geo_var": geo_var,   "geo_pct": 100 * geo_var  / total_var,
        "time_var": time_var, "time_pct": 100 * time_var / total_var,
    })

vd = pd.DataFrame(rows)
print("=== Variance decomposition (% of total) ===")
print(f"{'Variable':<10} {'Geographic %':<16} {'Temporal %':<14} {'Dominant'}")
print("-" * 55)
for _, r in vd.iterrows():
    dom = "GEOGRAPHIC" if r.geo_pct > r.time_pct else "TEMPORAL"
    print(f"{r.variable:<10} {r.geo_pct:<16.1f} {r.time_pct:<14.1f} {dom}")

=== Variance decomposition (% of total) ===
Variable   Geographic %     Temporal %     Dominant
-------------------------------------------------------
pdsi       23.7             76.3           TEMPORAL
air        31.6             68.4           TEMPORAL
prate      7.3              92.7           TEMPORAL


In [11]:
# Cell 10 — Uncertainty: within-run spread vs across-run std
print("=== Uncertainty (median across sample locations and all years) ===")
print(f"{'Variable':<10} {'Within-run spread':<22} {'Across-run std':<20} {'Ratio'}")
print("-" * 65)
for var in ["pdsi", "air", "prate"]:
    med_spread = np.nanmedian([np.nanmedian(s["spread"])     for s in series[var]])
    med_std    = np.nanmedian([np.nanmedian(s["std_across"])  for s in series[var]])
    ratio      = med_spread / med_std if med_std > 0 else float("nan")
    print(f"{var:<10} {med_spread:<22.4f} {med_std:<20.4f} {ratio:.2f}x")

# PDSI spread early vs late (proxy density effect)
print("=== PDSI within-run spread: early (0-500 CE) vs late (1500-1998 CE) ===")
early_mask = years < 500
late_mask  = years >= 1500
early_med = np.nanmedian([np.nanmean(s["spread"][early_mask]) for s in series["pdsi"]])
late_med  = np.nanmedian([np.nanmean(s["spread"][late_mask])  for s in series["pdsi"]])
print(f"  Early period median spread: {early_med:.4f}")
print(f"  Late  period median spread: {late_med:.4f}")
print(f"  Early/late ratio: {early_med/late_med:.2f}x  (>1 = more uncertain in early period)")

=== Uncertainty (median across sample locations and all years) ===
Variable   Within-run spread      Across-run std       Ratio
-----------------------------------------------------------------
pdsi       1.5145                 0.3269               4.63x
air        0.4823                 0.1116               4.32x
prate      0.0000                 0.0000               4.87x
=== PDSI within-run spread: early (0-500 CE) vs late (1500-1998 CE) ===
  Early period median spread: 1.5473
  Late  period median spread: 1.3642
  Early/late ratio: 1.13x  (>1 = more uncertain in early period)


In [13]:
# Cell 11 — Band C coherence check: LMR anomaly structure vs static BasinATLAS climatology
# LMR variables are anomalies (not absolute values), so direct unit comparison is impossible.
# Instead: check whether rank ordering is coherent.
# Late-period anomaly = LMR 1850-1900 mean minus LMR full-record mean (removes prior offset).

conn = db_connect()
late_mask = (years >= 1850) & (years <= 1900)

cmp_rows = []
for i, row in sample_df.iterrows():
    air_late  = float(np.nanmean(series["air"][i]["mean"][late_mask]))
    air_full  = float(np.nanmean(series["air"][i]["mean"]))
    air_anom  = air_late - air_full

    prate_late = float(np.nanmean(series["prate"][i]["mean"][late_mask]))
    prate_full = float(np.nanmean(series["prate"][i]["mean"]))
    prate_anom = prate_late - prate_full

    db_lon = float(row.lon) if float(row.lon) <= 180 else float(row.lon) - 360
    db_row = pd.read_sql("""
        SELECT tmp_dc_uyr / 10.0 AS tmp_c, pre_mm_uyr
        FROM public.basin08
        ORDER BY geom <-> ST_SetSRID(ST_MakePoint(%s, %s), 4326)
        LIMIT 1
    """, conn, params=(db_lon, float(row.lat)))

    if not db_row.empty:
        cmp_rows.append({
            "band": row["band"], "lat": row.lat, "lon": row.lon,
            "lmr_air_anom_K":    air_anom,
            "lmr_prate_anom":    prate_anom,
            "basin_tmp_C":       float(db_row["tmp_c"].iloc[0]),
            "basin_pre_mmyr":    float(db_row["pre_mm_uyr"].iloc[0]),
        })

conn.close()
cmp = pd.DataFrame(cmp_rows)

from scipy.stats import spearmanr
r_t, p_t = spearmanr(cmp["basin_tmp_C"],    cmp["lmr_air_anom_K"])
r_p, p_p = spearmanr(cmp["basin_pre_mmyr"], cmp["lmr_prate_anom"])

colors = [list(BAND_COLORS.values())[i % 7] for i in range(len(cmp))]
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (xc, yc, xl, yl, title) in zip(axes, [
    ("basin_tmp_C",    "lmr_air_anom_K",  "BasinATLAS temp (°C)",      "LMR 1850-1900 anomaly (K)",       "Temperature rank coherence"),
    ("basin_pre_mmyr", "lmr_prate_anom",  "BasinATLAS precip (mm/yr)", "LMR 1850-1900 anomaly (kg/m²/s)", "Precipitation rank coherence"),
]):
    ax.scatter(cmp[xc], cmp[yc], c=colors, s=60, zorder=3)
    ax.axhline(0, color="gray", lw=0.8, ls="--")
    ax.set_xlabel(xl); ax.set_ylabel(yl)
    ax.set_title(title)

from matplotlib.lines import Line2D
handles = [Line2D([0],[0], color=BAND_COLORS[b], marker="o", lw=0, markersize=7, label=b)
           for b in BAND_COLORS]
axes[0].legend(handles=handles, fontsize=7, loc="upper left")
plt.suptitle("LMR 1850-1900 anomaly (relative to full-record mean) vs Band C absolute climatology",
             fontsize=10)
plt.tight_layout()
plt.savefig(OUT_DIR / "10_lmr_bandc_comparison.png", dpi=120)
plt.show()

# Print and save after plt.show() to avoid Jupyter swallowing output
print(f"Temperature rank coherence  (Band C absolute vs LMR 1850-1900 anomaly): Spearman r = {r_t:.3f}  p = {p_t:.3f}")
print(f"Precipitation rank coherence (Band C absolute vs LMR 1850-1900 anomaly): Spearman r = {r_p:.3f}  p = {p_p:.3f}")
print(f"\nLMR air 1850-1900 anomaly stats (K, relative to 2000-yr mean):")
print(cmp["lmr_air_anom_K"].describe().round(4).to_string())

cmp.to_csv(OUT_DIR / "10_bandc_comparison.csv", index=False)
print(f"\nSaved 10_bandc_comparison.csv")

Temperature rank coherence  (Band C absolute vs LMR 1850-1900 anomaly): Spearman r = -0.055  p = 0.759
Precipitation rank coherence (Band C absolute vs LMR 1850-1900 anomaly): Spearman r = -0.112  p = 0.527

LMR air 1850-1900 anomaly stats (K, relative to 2000-yr mean):
count    34.0000
mean     -0.0648
std       0.0748
min      -0.2210
25%      -0.1041
50%      -0.0528
75%      -0.0103
max       0.0613

Saved 10_bandc_comparison.csv


In [17]:
# Cell 12 — L8 to LMR cell mapping: how many basins share each 2x2 deg cell?
conn = db_connect()
basins = pd.read_sql(
    "SELECT hybas_id, ST_X(ST_Centroid(geom)) AS lon, ST_Y(ST_Centroid(geom)) AS lat FROM public.basin08",
    conn)
conn.close()

basins["lmr_lat"] = (np.round(basins["lat"].values / 2) * 2).clip(-90, 90)
basins["lmr_lon"] =  np.round(basins["lon"].values / 2) * 2
basins["lmr_cell"] = basins["lmr_lat"].astype(str) + "_" + basins["lmr_lon"].astype(str)

cell_counts = basins["lmr_cell"].value_counts()

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(cell_counts.values, bins=60, log=True, color="steelblue", edgecolor="white")
ax.set_xlabel("L8 basins per LMR 2x2 deg cell")
ax.set_ylabel("Count of LMR cells (log scale)")
ax.set_title("L8 to LMR cell mapping: basin sharing per grid cell")
plt.tight_layout()
plt.savefig(OUT_DIR / "10_lmr_basin_sharing.png", dpi=120)
plt.show()

# Stats after plt.show() to avoid being swallowed
print(f"Total L8 basins: {len(basins):,}")
print(f"Unique LMR cells occupied: {len(cell_counts):,}  (of {n_valid:,} valid land cells)")
print(f"LMR land cells with no L8 basin: {n_valid - len(cell_counts):,}")
print(f"\nBasins-per-cell distribution:")
for stat, val in [("median", cell_counts.median()), ("p75", cell_counts.quantile(0.75)),
                  ("p95", cell_counts.quantile(0.95)), ("max", cell_counts.max())]:
    print(f"  {stat}: {val:.0f}")

stats = pd.DataFrame([{
    "total_l8_basins": len(basins),
    "unique_lmr_cells": len(cell_counts),
    "lmr_cells_no_basin": n_valid - len(cell_counts),
    "basins_per_cell_median": cell_counts.median(),
    "basins_per_cell_p75": cell_counts.quantile(0.75),
    "basins_per_cell_p95": cell_counts.quantile(0.95),
    "basins_per_cell_max": cell_counts.max(),
}])
stats.to_csv(OUT_DIR / "10_basin_sharing_stats.csv", index=False)
print("\nSaved 10_basin_sharing_stats.csv")

Total L8 basins: 190,675
Unique LMR cells occupied: 4,999  (of 16,380 valid land cells)
LMR land cells with no L8 basin: 11,381

Basins-per-cell distribution:
  median: 39
  p75: 56
  p95: 74
  max: 109

Saved 10_basin_sharing_stats.csv


In [16]:
# Cell 13 — Save outputs
vd.to_csv(OUT_DIR / "10_variance_decomposition.csv", index=False)
sample_df.to_csv(OUT_DIR / "10_sample_locations.csv", index=False)
cmp.to_csv(OUT_DIR / "10_bandc_comparison.csv", index=False)

print("Outputs saved:")
for fname in ["10_variance_decomposition.csv", "10_sample_locations.csv", "10_bandc_comparison.csv",
              "10_lmr_land_grid.png", "10_lmr_sample_timeseries.png",
              "10_lmr_bandc_comparison.png", "10_lmr_basin_sharing.png"]:
    p = OUT_DIR / fname
    size = p.stat().st_size if p.exists() else 0
    print(f"  {fname}  ({size/1024:.0f} KB)")

Outputs saved:
  10_variance_decomposition.csv  (0 KB)
  10_sample_locations.csv  (1 KB)
  10_bandc_comparison.csv  (3 KB)
  10_lmr_land_grid.png  (53 KB)
  10_lmr_sample_timeseries.png  (892 KB)
  10_lmr_bandc_comparison.png  (85 KB)
  10_lmr_basin_sharing.png  (25 KB)
